In [ ]:
# CELL 1 - Nap file Bitstream (.bit) vao FPGA PYNQ-Z2
# Khong can file .hwh; Python phai chay tren board PYNQ-Z2.
from pathlib import Path
from pynq import Bitstream, MMIO

BIT_NAME = "design_1_wrapper.bit"
# Uu tien .bit cung thu muc notebook; hoac trong Bitstream/ o thu muc cha.
candidates = [Path.cwd() / BIT_NAME, Path.cwd().parent / "Bitstream" / BIT_NAME]
bit_path = next((p for p in candidates if p.exists()), None)
if bit_path is None:
    raise FileNotFoundError("Khong tim thay design_1_wrapper.bit. Hay upload file .bit len board.")

Bitstream(str(bit_path)).download()  # Cau hinh phan PL cua SoC
print("Da nap Bitstream:", bit_path)


In [ ]:
# CELL 2 - Dieu khien LED bang MMIO (thanh ghi AXI4-Lite)
# slv_reg0 o offset 0x00; bon bit thap dieu khien LD0, LD1, LD2, LD3.
BASE_ADDR = 0x43C00000  # Base Address MY_IP_0 trong Vivado
REG_OFFSET = 0x00       # slv_reg0
led = MMIO(BASE_ADDR, 0x1000)

LED_VALUE = 0b1111     # Sua: 0000=tat het, 0001=LD0, 0101=LD0+LD2, 1111=bat het
led.write(REG_OFFSET, LED_VALUE)
value = led.read(REG_OFFSET) & 0xF
print("Da ghi:", format(LED_VALUE, "04b"), "| Doc lai:", format(value, "04b"))
# Gia tri doc lai cho biet thanh ghi; can quan sat board de kiem tra LED that.


In [ ]:
# CELL 3 - Mo website ngay trong Jupyter, KHONG CAN thu muc Website/
# Can goi Flask cai trong moi truong Python dang chay notebook.
# Sau khi chay cell nay, mo http://192.168.2.99:5000 tren PC.
from flask import Flask, request, jsonify
from threading import Thread
from werkzeug.serving import make_server

app = Flask(__name__)
HTML = """<!doctype html>
<html lang="vi">
<head>
  <meta charset="utf-8">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>PYNQ-Z2 LED Control</title>
  <style>
    body {font-family:Arial,sans-serif; text-align:center; margin:0; padding:34px 12px; background:#fff;}
    h1 {font-size:32px; margin:15px 0 26px;}
    button {border:2px solid #aaa; border-radius:11px; background:#f2f2f2; padding:17px 22px; margin:5px; font-size:16px; cursor:pointer;}
    #state {font-size:20px; margin-top:25px;}
  </style>
</head>
<body>
  <h1>PYNQ-Z2 LED Control</h1>
  <div id="buttons"></div>
  <div>
    <button onclick="sendLED(15)">BAT TAT CA</button>
    <button onclick="sendLED(0)">TAT TAT CA</button>
  </div>
  <p id="state">LED: ----</p>
  <script>
    let current = 0;
    const buttons = document.getElementById('buttons');
    // Tao 4 nut LD0..LD3, nhan moi nut se dao bit LED tuong ung.
    for (let i = 0; i < 4; i++) {
      const btn = document.createElement('button');
      btn.textContent = 'LD' + i;
      btn.onclick = () => sendLED(current ^ (1 << i));
      buttons.appendChild(btn);
    }
    function showState() {
      document.getElementById('state').textContent =
        'LED: ' + current.toString(2).padStart(4, '0');
    }
    async function sendLED(value) {
      try {
        const response = await fetch('/set', {
          method:'POST',
          headers:{'Content-Type':'application/json'},
          body:JSON.stringify({value:value})
        });
        if (!response.ok) throw new Error('HTTP ' + response.status);
        current = (await response.json()).value;
        showState();
      } catch(err) {
        document.getElementById('state').textContent = 'Loi ket noi: ' + err.message;
      }
    }
    fetch('/state').then(r=>r.json()).then(d=>{
      current=d.value; showState();
    }).catch(()=>{
      document.getElementById('state').textContent='Khong doc duoc FPGA';
    });
  </script>
</body>
</html>"""

@app.get('/')
def home():
    return HTML

@app.get('/state')
def get_led_state():
    return jsonify(value=led.read(REG_OFFSET) & 0xF)

@app.post('/set')
def set_led_state():
    data = request.get_json(silent=True) or {}
    value = data.get('value')
    if type(value) is not int or not (0 <= value <= 15):
        return jsonify(error='value phai la so nguyen tu 0 den 15'), 400
    led.write(REG_OFFSET, value)
    return jsonify(value=led.read(REG_OFFSET) & 0xF)

# Chay web server o thread rieng de khong chiem kernel Jupyter.
# Chay lai Cell 3 se tat server cu truoc khi mo server moi.
if '_led_web_server' in globals():
    _led_web_server.shutdown()
    _led_web_server.server_close()
_led_web_server = make_server('0.0.0.0', 5000, app, threaded=True)
_led_web_thread = Thread(target=_led_web_server.serve_forever, daemon=True)
_led_web_thread.start()
print('Website: http://192.168.2.99:5000')
# Website dieu khien BAT/TAT, chua co mach dem xung FPGA tu nhap nhay.
